## Concept 4 — MessagesPlaceholder & Conversation History

### What is it?
`MessagesPlaceholder` reserves a slot in your prompt template where a dynamic list of past messages
is injected at runtime. This is how multi-turn conversations are built.

### Pipeline
```
[System] + [past messages injected here] + [new HumanMessage] → LLM → AIMessage
                ↑
           MessagesPlaceholder
```

### Limitation (what Concept 5 fixes)
LLM returns unstructured text — hard to parse programmatically.
→ Concept 5 forces the LLM to return a Pydantic schema or JSON.

In [ ]:
print('Hi')

In [ ]:
import sys
sys.path.insert(0, '.')

from PromptUtils import get_llm
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.output_parsers import StrOutputParser

llm = get_llm()
print('LLM ready')

### Step 1 — Why History Matters
Without history the LLM forgets everything between turns.

In [ ]:
# No history — LLM treats each call independently
no_history_prompt = ChatPromptTemplate.from_messages([
    ('system', 'You are a helpful assistant.'),
    ('human',  '{input}'),
])
chain = no_history_prompt | llm | StrOutputParser()

r1 = chain.invoke({'input': 'My name is Mohan.'})
print(f'Turn 1: {r1}')

r2 = chain.invoke({'input': 'What is my name?'})
print(f'Turn 2: {r2}')  # Forgets — has no idea

### Step 2 — MessagesPlaceholder
Reserve a slot for history. Inject past messages as a list at runtime.

In [ ]:
history_prompt = ChatPromptTemplate.from_messages([
    ('system', 'You are a helpful assistant.'),
    MessagesPlaceholder(variable_name='history'),  # slot for past messages
    ('human', '{input}'),
])

# Inspect what the prompt looks like with history injected
sample_history = [
    HumanMessage(content='My name is Mohan.'),
    AIMessage(content='Hello Mohan! How can I help you?'),
]

msgs = history_prompt.format_messages(history=sample_history, input='What is my name?')
print(f'{len(msgs)} messages in prompt:')
for m in msgs:
    print(f'  [{m.type}] {m.content}')

### Step 3 — Full Conversation Loop (Full History)
Every turn appends to `history`. The LLM sees the entire conversation.

In [ ]:
history_chain = history_prompt | llm | StrOutputParser()
history = []

def chat(user_input):
    response = history_chain.invoke({'history': history, 'input': user_input})
    history.append(HumanMessage(content=user_input))
    history.append(AIMessage(content=response))
    return response

# Simulate a multi-turn conversation
turns = [
    'My name is Mohan and I am a data scientist.',
    'What is my job?',
    'Can you summarize what you know about me?',
]

for turn in turns:
    print(f'Human: {turn}')
    print(f'AI:    {chat(turn)}')
    print(f'History length: {len(history)} messages\n')

### Step 4 — Window Memory Strategy
Keep only the last N messages to prevent the context window from growing too large.

In [ ]:
WINDOW_SIZE = 4  # keep last 4 messages (2 turns)
window_history = []

def chat_windowed(user_input):
    # Use only last WINDOW_SIZE messages
    recent = window_history[-WINDOW_SIZE:]
    response = history_chain.invoke({'history': recent, 'input': user_input})
    window_history.append(HumanMessage(content=user_input))
    window_history.append(AIMessage(content=response))
    print(f'  [using {len(recent)} of {len(window_history)-2} stored messages]')
    return response

window_turns = [
    'I am learning Python.',
    'I started last month.',
    'I struggle with decorators.',
    'What have I told you about myself?',  # window may drop turn 1
]

for t in window_turns:
    print(f'Human: {t}')
    print(f'AI:    {chat_windowed(t)}')
    print()

### Full Function

In [ ]:
session_history = []

def conversation(user_input):
    response = history_chain.invoke({'history': session_history, 'input': user_input})
    session_history.append(HumanMessage(content=user_input))
    session_history.append(AIMessage(content=response))
    return response

print('Chat session (type questions below):')
demo = ['My favourite language is Python.', 'What language did I mention?']
for q in demo:
    print(f'Q: {q}')
    print(f'A: {conversation(q)}\n')